
# GRAHSP Fig. 7 reproduction: attenuation of the galaxy model

Reproduction of Fig. 7 of Buchner et al. (2024, GRAHSP): a star-forming
galaxy SED from intrinsic (dark blue) to strongly attenuated (dark red) as the
diffuse color excess E(B-V) is swept from 0.01 to 10. Energy balance routes
the attenuated UV/optical light into the far-IR dust bump (Dale 2014), so the
curves pivot about the FIR peak while the UV is progressively suppressed.

The extremely attenuated, low-metallicity starbursting galaxy **Haro 11**
(photometry from NED, mirroring Lyu et al. 2016) is overplotted as a dashed
red curve for reference.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import itertools
import warnings
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm, colors

import tengri
from tengri import DEFAULT, Fixed, SEDModel
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_NM_HZ = 2.99792458e17
R_V = 4.05  # Calzetti: A_V = R_V * E(B-V); tau_V = A_V / 1.086

# Resolve the committed Haro 11 NED SED next to this script. sphinx-gallery
# executes the example without a ``__file__`` in scope, so fall back to the
# repo-anchored path via the installed tengri package (editable layout).
if "__file__" in dir():
    _data_dir = Path(__file__).resolve().parent / "data"
else:
    _repo = Path(tengri.__file__).resolve().parents[2]
    _data_dir = _repo / "examples" / "dust_attenuation" / "data"
HARO11 = _data_dir / "haro11_ned_sed.txt"

# No-arg load_ssp() auto-discovers the bundled PARSEC/MILES/Chabrier wNE grid
# regardless of the working directory (sphinx-gallery executes from elsewhere).
ssp = tengri.load_ssp()
wave_aa = jnp.logspace(np.log10(800.0), np.log10(1.0e7), 3000)  # 0.08 - 1000 um
wave_um = np.asarray(wave_aa) / 1e4

ebv_grid = np.logspace(np.log10(0.01), np.log10(10.0), 9)
norm = colors.LogNorm(vmin=0.01, vmax=10.0)
cmap = plt.get_cmap("RdBu_r")


def nu_Lnu(lnu):
    return np.asarray(lnu) * (C_NM_HZ / (np.asarray(wave_aa) * 0.1))


fig, ax = plt.subplots(figsize=(6.6, 7.6))

# Build the SED model ONCE; the diffuse-screen optical depth ``dust_tau_diff``
# is a parameter, so the E(B-V) sweep only varies that key in the fixed-value
# dict and re-runs the (already-compiled) forward pass. Rebuilding the full
# stellar+dust model inside the loop would recompile the SSP pipeline on every
# iteration and accumulate XLA buffers — the gallery-OOM anti-pattern.
model = SEDModel.build(
    ssp_data=ssp,
    sfh={
        "type": "delayed",
        "all_params": Fixed(DEFAULT),
        "tau_gyr": 5.0,
        "age_gyr": 3.0,
        "log_total_mass": 10.0,
    },
    dust_attenuation={
        "type": "two_component",
        "law": "calzetti",
        "all_params": Fixed(DEFAULT),
        "tau_bc": 0.3,  # fixed birth-cloud baseline (stabilizes the FIR peak)
        "tau_diff": 0.3,  # baseline; overridden per E(B-V) below
    },
    dust_emission={"type": "dale2014", "all_params": Fixed(DEFAULT)},
    redshift=Fixed(0.01),
)
base_params = model.spec.get_fixed_values()

norm_ref = None
for ebv in ebv_grid:
    tau_diff = R_V * ebv / 1.086  # diffuse screen scales with E(B-V)
    params = {**base_params, "dust_tau_diff": tau_diff}
    rest = model.predict(params)
    lnu = np.asarray(rest.rest_sed(np.asarray(wave_aa)))
    lflam = nu_Lnu(lnu)
    if norm_ref is None:
        # Normalize so the FIR dust peak sits near ~5 (paper scaling).
        fir = (wave_um > 30) & (wave_um < 300)
        norm_ref = lflam[fir].max() / 5.0
    ax.plot(wave_um, lflam / norm_ref, color=cmap(norm(ebv)), lw=1.4, zorder=3)

# Haro 11 overlay: raw NED photometry has many measurements per band (plus
# radio + upper limits), so bin into log-wavelength medians for a smooth,
# representative SED (cf. the Lyu+ 2016 model curve in the paper).
if HARO11.exists():
    h = np.loadtxt(HARO11)
    hw, hf = h[:, 0], h[:, 1]
    good = np.isfinite(hf) & (hf > 0) & (hw > 0.1) & (hw < 500.0)
    hw, hf = hw[good], hf[good]
    edges = np.logspace(np.log10(0.1), np.log10(500.0), 20)
    centers, meds = [], []
    for lo, hi in itertools.pairwise(edges):
        sel = (hw >= lo) & (hw < hi)
        if sel.sum() >= 2:  # require >=2 measurements to suppress single outliers
            centers.append(np.sqrt(lo * hi))
            meds.append(np.median(hf[sel]))
    centers, meds = np.array(centers), np.array(meds)
    # Despike: 2 passes dropping bins >0.5 dex from a 3-point rolling median.
    for _ in range(2):
        logm = np.log10(meds)
        roll = np.array([np.median(logm[max(0, i - 1) : i + 2]) for i in range(len(logm))])
        keep = np.abs(logm - roll) < 0.5
        centers, meds = centers[keep], meds[keep]
    fir = (centers > 40) & (centers < 200)
    if fir.any():
        meds = meds / (np.median(meds[fir]) / 5.0)
    # Final guard: in lambda*F_lambda a dusty starburst cannot exceed its FIR
    # bump, so drop residual NED outlier bins above 3x the FIR peak (~5).
    ok = meds < 15.0
    centers, meds = centers[ok], meds[ok]
    ax.plot(
        centers,
        meds,
        color="red",
        ls="--",
        lw=1.8,
        marker="o",
        ms=3.5,
        label="Haro 11 (NED)",
        zorder=5,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.1, 1000.0)
ax.set_ylim(1e-3, 1e2)
ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"$\lambda F_\lambda$ [arb.]")

secax = ax.secondary_xaxis(
    "top", functions=(lambda x: C_NM_HZ / 1e3 / x, lambda nu: C_NM_HZ / 1e3 / nu)
)
secax.set_xlabel("Frequency [Hz]")

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.02)
cbar.set_label("E(B-V)")
cbar.set_ticks([0.01, 0.3, 10.0])
cbar.set_ticklabels(["0.01", "0.3", "10"])

fig.tight_layout()
plt.savefig("plot_grahsp_paper_fig7_galaxy_attenuation.png", dpi=150, bbox_inches="tight")